# A third opinion: LLM-based hallucination classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_LLMVerdict.ipynb)

`Colab_HallucinationTaxonomy.ipynb` classifies hypotheses with thresholded text statistics. Atwany
et al. (ACL Findings 2025) do the same job by asking an LLM to compare reference against hypothesis,
and report the **Hallucination Error Rate (HER)** — hallucination errors over total examples. Their
paper publishes the prompt verbatim (Figure 5, coarse-grained; Figure 6, fine-grained), so the
method is reproducible without any code from them.

This notebook runs that classifier over the same 20 000 hypotheses and asks the question the
taxonomy alone cannot answer: **does an independent method agree with it, and does it agree about
the shape across model scale?**

Three reasons this is worth the run rather than a curiosity:

- **It is a genuinely different paradigm.** The taxonomy thresholds surface statistics; the LLM reads
  the pair. Where they agree, the scale trend is not an artifact of five hand-chosen thresholds.
- **Atwany et al. report the agreement structure to compare against.** Human–human 0.71,
  human–GPT 0.60, human–Gemini 0.59, GPT–Gemini 0.78 — and human–heuristic **0.00**, for a
  threshold heuristic much like ours. That last number is the one this notebook exists to test
  against our detector rather than assume away.
- **Two models, not one.** Their cross-model agreement (0.78) exceeded either model's agreement with
  humans, which is what makes substituting a different LLM defensible — and what makes running two
  and reporting their agreement better than running one and trusting it.

### Licensing: a deliberate decision, recorded here

Every other notebook in this project keeps TIMIT reference text off the wire — digests instead of
transcripts, numbers-only files, no text in printed output. **This notebook breaks that rule
knowingly**: classifying a hypothesis against its reference requires sending both to a third-party
API, and the full-grid run transmits all 1000 references. That was an explicit choice, not an
oversight, and it is recorded in `llm_provenance.json` so a later reader sees the decision rather
than inferring it.

TIMIT is LDC93S1 — licensed, not redistributable. Confirm your LDC terms permit this before running
section 5. Nothing leaves the machine until that cell executes.

## 1. Inputs, cost, and auth

| input | source |
|---|---|
| `delta_results_full.csv` | Drive — the 20 000 hypotheses |
| `delta_provenance.json` | Drive — `reference_digest`, to verify the references |
| `halluc_taxonomy.csv` | Drive — the taxonomy verdict, for the agreement analysis |
| TIMIT `TEST/**/*.TXT` | Drive — reference transcripts |
| `corpus_digests.json` | GitHub — the draw order |

**Two models, for the same reason Atwany et al. used two.** `claude-opus-5` is the primary; the
`claude-haiku-4-5` arm is a capability contrast *and* the closer replication of their setup — Haiku
still accepts `temperature`, so that arm can run the greedy decoding their §4.3 specifies, while
Opus 5 rejects the parameter outright.

Cost is dominated by three choices, all made in section 3 and measured in section 4:

- **Batch API** halves everything. 20 000 classifications is not latency-sensitive.
- **Prompt caching** on the instruction block, which is identical across all 20 000 calls. Cache
  reads bill at 0.1x input. The minimum cacheable prefix is **512 tokens on Opus 5 and 4096 on
  Haiku 4.5**, so section 4 measures the block and reports which arms will actually cache rather
  than assuming.
- **Thinking off.** A three-way classification does not need it, output tokens are the expensive
  side on Opus 5, and their GPT-4o-mini baseline had no thinking either — so disabling it is both
  cheaper and the more faithful replication.

In [ ]:
!pip -q install anthropic

import collections, csv, glob, hashlib, json, math, os, platform, time
import numpy as np
import anthropic

from google.colab import drive
drive.mount("/content/drive")

# API key: Colab secret `ANTHROPIC_API_KEY` (recommended) or a prompt
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

client = anthropic.Anthropic()

DRIVE_ROOT = "/content/drive/MyDrive/NAACL"
FULL_CSV   = os.path.join(DRIVE_ROOT, "delta_results_full.csv")     # hypotheses
PROV_IN    = os.path.join(DRIVE_ROOT, "delta_provenance.json")      # reference_digest
TAX_CSV    = os.path.join(DRIVE_ROOT, "halluc_taxonomy.csv")        # taxonomy verdict
BATCH_JSON = os.path.join(DRIVE_ROOT, "llm_batches.json")           # submitted batch ids
OUT_CSV    = os.path.join(DRIVE_ROOT, "llm_verdict_per_utterance.csv")
PROV_JSON  = os.path.join(DRIVE_ROOT, "llm_provenance.json")

MODELS = ["tiny", "base", "small", "medium", "large-v3"]            # whisper ladder
PARAMS = {"tiny": "37.2M", "base": "71.8M", "small": "240.6M",
          "medium": "762.3M", "large-v3": "1541.6M"}
CONDS  = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]
HEAD   = (25, "on")

# the judges. opus-5 caches at >=512 prompt tokens; haiku-4-5 needs >=4096 (measured in section 4)
JUDGES     = ["claude-opus-5", "claude-haiku-4-5"]
CHUNK      = 5000        # requests per batch submission; 4 chunks x 20000 rows per judge
MAX_TOKENS = 256         # schema-constrained label; generous headroom, billed only on use

## 2. Load, verify, and join

Same integrity checks the taxonomy notebook runs, for the same reason: a classification is only
meaningful against the corpus that produced it. The references are rebuilt from TIMIT and hashed
against the `reference_digest` the delta sweep pinned — if the pin mismatches, the LLM would be
comparing hypotheses against the wrong sentences and every verdict below would be noise.

The taxonomy verdict is joined here too, so section 8 can compute agreement without re-reading
anything.

In [ ]:
def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


for p in (FULL_CSV, PROV_IN, TAX_CSV):
    assert os.path.exists(p), f"missing input: {p}"
INPUT_DIGESTS = {os.path.basename(p): sha256_file(p) for p in (FULL_CSV, PROV_IN, TAX_CSV)}

full    = list(csv.DictReader(open(FULL_CSV, newline="")))
tax     = list(csv.DictReader(open(TAX_CSV,  newline="")))
prov_in = json.load(open(PROV_IN))

cells = collections.Counter((r["model"], int(r["offset_s"]), r["timestamps"]) for r in full)
assert set(cells) == {(m, o, t) for m in MODELS for o, t in CONDS}, "grid has holes"
assert set(cells.values()) == {1000}, sorted(set(cells.values()))
assert sum("<|" in r["text"] for r in full) == 0, "special tokens leaked into text"

TAX = {(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]): r for r in tax}
assert len(TAX) == len(full), (len(TAX), len(full))

# --- references, verified against the delta sweep's pin --------------------------------
cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
FILES = json.load(open("corpus_digests.json"))["files"][0:1000]

from whisper.normalizers import EnglishTextNormalizer          # noqa: E402
normalizer = EnglishTextNormalizer()


def load_reference(wav_path):
    """TIMIT .TXT is `<start_sample> <end_sample> <sentence>` -- the integers are not words."""
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]


REF_RAW = {r["path"]: load_reference(os.path.join(TIMIT_TEST, r["path"])) for r in FILES}
REF     = {p: normalizer(t) for p, t in REF_RAW.items()}
rebuilt = sha256_bytes("\n".join(f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())

pinned = None
for block in prov_in.values():
    if isinstance(block, dict) and "reference_digest" in block:
        pinned = block["reference_digest"]
        break
assert pinned is not None and rebuilt == pinned, (
    "reference digest mismatch (or absent): this TIMIT copy is not the corpus experiment A "
    "decoded, so every verdict below would score against the wrong sentences")

print(f"{len(full)} hypotheses | {len(REF)} references | digest matches the delta sweep's pin")
for k, v in INPUT_DIGESTS.items():
    print(f"  {k:<28} sha256 {v[:16]}...")

## 3. The prompt, and three deliberate deviations

The instruction text below is Atwany et al.'s Figure 5 coarse-grained prompt, transcribed verbatim
including all six examples. Three things about *how* it is sent differ from the paper, each for a
stated reason:

**1. Split across the system/user boundary.** The paper presents one block ending with an `Input:`
template. Here the instructions and examples live in `system` and only the reference/hypothesis pair
goes in the user turn. The rendered text and its order are unchanged — `system` renders before
`messages` — but the split is what makes the instruction block a cacheable stable prefix. Without
it, every one of the 20 000 calls pays full input price for the same 800 tokens.

**2. Structured outputs replace "produce only the classification".** The paper constrains the output
by asking; `output_config.format` with an enum schema constrains it by construction. This removes
the parse step entirely and forecloses the two failure modes a prose instruction cannot: a preamble
before the label, and stray internal XML in the response. It is a strict improvement on the method,
not a departure from it.

**3. Thinking off, and only one arm can be greedy.** Their §4.3 specifies greedy decoding for
reproducibility. `temperature` is rejected outright on Claude Opus 5, so that arm cannot replicate
it — the `haiku-4-5` arm can and does, which is a second reason to run both. Thinking is disabled on
the Opus arm (permitted at effort `high` or below), matching their non-thinking GPT-4o-mini baseline
and keeping output tokens at roughly ten per call instead of several hundred.

The fine-grained prompt (their Figure 6, five categories) is transcribed too but not run by default
— `GRAIN` selects it. Coarse is the default because the agreement analysis in section 8 needs a
binary hallucination/not verdict to compare against the taxonomy.

In [ ]:
COARSE_LABELS = ["Hallucination Error", "Non-Hallucination Error", "No Error"]
FINE_LABELS   = ["Phonetic Error", "Oscillation Error", "Hallucination Error",
                 "Language Error", "No Error"]

COARSE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth. This includes: - Fabricated Content: Words or phrases \
entirely absent in the ground truth. - Meaningful Contradictions: Significant changes in the \
meaning from the ground truth. - Invented Context: Introduction of details or context not present \
in the ground truth. - Note: These errors involve fabrication of new information or significant \
distortion of meaning, beyond grammatical or structural mistakes.
2. Non-Hallucination Error: Errors that do not involve fabrication or significant contradictions \
of the ground truth. These include: - Phonetic Errors: Substitutions of phonetically similar words \
or minor pronunciation differences. - Structural or Language Errors: Grammatical, syntactic, or \
structural issues that make the text incoherent or incorrect (e.g., incorrect verb tenses, \
subject-verb agreement problems, omissions, or insertions). - Oscillation Errors: Repetitive, \
nonsensical patterns or sounds that do not convey linguistic meaning (e.g., "ay ay ay ay"). \
- Other Non-Hallucination Errors: Errors that do not fit the above subcategories but are not \
hallucinations.
3. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs. Minor differences in wording, phrasing, or grammar that \
do not alter the intended meaning are acceptable.

Input Format:
Ground Truth: The original, accurate text provided.
Generated Output: The text produced by the speech recognition system.

Output Format: Classify the input text pairs into one of the following:
Non-Hallucination Error
Hallucination Error
No Error

Examples:
Example 1:
Ground Truth: "A millimeter roughly equals one twenty-fifth of an inch."
Generated Output: "Miller made her roughly one twenty-fifths of an inch."
Output: Non-Hallucination Error

Example 2:
Ground Truth: "Indeed, ah!"
Generated Output: "Ay ay indeed ay ay ay ay ay ay."
Output: Non-Hallucination Error

Example 3:
Ground Truth: "Captain Lake did not look at all like a London dandy now."
Generated Output: "Will you let Annabel ask her if she sees what it is you hold in your arms \
again?"
Output: Hallucination Error

Example 4:
Ground Truth: "The patient was advised to take paracetamol for fever and rest for two days."
Generated Output: "The patient was advised to take amoxicillin for fever and undergo surgery \
immediately."
Output: Hallucination Error

Example 5:
Ground Truth: "I need to book a flight to New York."
Generated Output: "I need to book ticket to New York."
Output: No Error

Example 6:
Ground Truth: "She went to the store yesterday."
Generated Output: "She went to the shop yesterday."
Output: No Error

Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

FINE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Phonetic Error: The output contains substitutions of phonetically similar words that do not \
match the ground truth and do not introduce broader grammatical or structural issues.
2. Oscillation Error: The output includes repetitive, nonsensical patterns or sounds that do not \
convey linguistic meaning (e.g., "ay ay ay ay").
3. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth.
4. Language Error: The output includes grammatical, syntactic, or structural issues that make the \
text incoherent or linguistically incorrect.
5. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs.

Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

GRAIN  = "coarse"                                   # "coarse" (default) or "fine"
SYSTEM = COARSE_SYSTEM if GRAIN == "coarse" else FINE_SYSTEM
LABELS = COARSE_LABELS if GRAIN == "coarse" else FINE_LABELS

SCHEMA = {"type": "object",
          "properties": {"label": {"type": "string", "enum": LABELS}},
          "required": ["label"], "additionalProperties": False}


def user_turn(ref, hyp):
    return f'Ground Truth: "{ref}"\nGenerated Output: "{hyp}"'


def params_for(judge, ref, hyp):
    """Per-judge request params. Opus 5 rejects `temperature` and takes `effort`; Haiku 4.5
    accepts `temperature` (so that arm runs the paper's greedy decoding) and rejects `effort`."""
    p = {
        "model": judge,
        "max_tokens": MAX_TOKENS,
        "system": [{"type": "text", "text": SYSTEM,
                    "cache_control": {"type": "ephemeral", "ttl": "1h"}}],
        "messages": [{"role": "user", "content": user_turn(ref, hyp)}],
        "output_config": {"format": {"type": "json_schema", "schema": SCHEMA}},
    }
    if judge.startswith("claude-opus-5"):
        p["thinking"] = {"type": "disabled"}         # allowed at effort high or below
        p["output_config"]["effort"] = "low"
    else:
        p["temperature"] = 0.0                       # the paper's greedy decoding, verbatim
    return p


print(f"grain={GRAIN} | {len(LABELS)} labels | system block {len(SYSTEM)} chars")
print("labels:", ", ".join(LABELS))

## 4. Measure before spending

Two things are worth knowing before submitting 40 000 requests, and both are cheap to check.

**Does the instruction block actually cache?** The minimum cacheable prefix is 512 tokens on
Claude Opus 5 and 4096 on Haiku 4.5. Below the threshold, caching is not an error — it silently does
nothing, and `cache_creation_input_tokens` stays 0. Measuring the block with `count_tokens` and
comparing against each judge's minimum is the difference between an $8 run and a $17 one on the
Opus arm, and it is the honest way to state the cost rather than assuming.

**What will it cost?** The projection below prices each arm at list rates with the Batch API's 50%
discount applied, splitting input into the cached prefix (0.1x) and the per-row remainder. Atwany
et al.'s own accounting — roughly 1000 tokens per example, under five output tokens — scales to
about \$1.56 for a grid this size on GPT-4o-mini; the figures here should land in the same order of
magnitude, and a projection far outside it means something is wrong with the request shape.

In [ ]:
PRICE = {                                            # USD per million tokens, list rates
    "claude-opus-5":    {"in": 5.00, "out": 25.00, "cache_min": 512},
    "claude-haiku-4-5": {"in": 1.00, "out":  5.00, "cache_min": 4096},
}
BATCH_DISCOUNT, CACHE_READ, CACHE_WRITE = 0.50, 0.10, 1.25

_ref, _hyp = REF[FILES[0]["path"]], normalizer(full[0]["text"])
sys_tok = client.messages.count_tokens(
    model=JUDGES[0], system=[{"type": "text", "text": SYSTEM}],
    messages=[{"role": "user", "content": ""}]).input_tokens
all_tok = client.messages.count_tokens(
    model=JUDGES[0], system=[{"type": "text", "text": SYSTEM}],
    messages=[{"role": "user", "content": user_turn(_ref, _hyp)}]).input_tokens
var_tok, out_tok, N = all_tok - sys_tok, 12, len(full)

print(f"system block {sys_tok} tokens | per-row variable ~{var_tok} | ~{out_tok} out | N = {N}\n")
hdr = f"{'judge':>18} {'caches?':>8} {'input $':>9} {'output $':>9} {'batch total $':>14}"
print(hdr + "\n" + "-" * len(hdr))
TOTAL = 0.0
for j in JUDGES:
    p = PRICE[j]
    caches = sys_tok >= p["cache_min"]
    if caches:
        cin = (sys_tok * CACHE_WRITE + N * sys_tok * CACHE_READ + N * var_tok) / 1e6 * p["in"]
    else:
        cin = N * (sys_tok + var_tok) / 1e6 * p["in"]
    cout = N * out_tok / 1e6 * p["out"]
    tot  = (cin + cout) * BATCH_DISCOUNT
    TOTAL += tot
    print(f"{j:>18} {('yes' if caches else 'NO'):>8} {cin:>9.2f} {cout:>9.2f} {tot:>14.2f}")
print("-" * len(hdr))
print(f"{'both arms':>18} {'':>8} {'':>9} {'':>9} {TOTAL:>14.2f}")
print(f"\nAtwany et al. scale to ~$1.56 for {N} rows on GPT-4o-mini (their App. A.4.1 rate card).")
print("A projection far outside this order of magnitude means the request shape is wrong.")

## 5. Submit — the cell that sends reference text off this machine

**This is the licensing boundary.** Running it transmits all 1000 TIMIT reference transcripts, plus
every hypothesis, to the Anthropic API. Nothing above this point has left the runtime.

Submission is chunked at `CHUNK` rows and every batch id is written to `llm_batches.json` on Drive
*before* the next chunk goes out. Batches are durable server-side for 29 days, so a disconnected
runtime costs nothing: re-running this cell re-reads the id file and submits only what is missing,
and section 6 collects results from ids alone. `custom_id` is the row's index into `full`, which is
stable because `full` comes from a digest-verified file.

In [ ]:
CONFIRM = False        # set True to send reference + hypothesis text to the Anthropic API

assert CONFIRM, ("set CONFIRM = True to proceed. This transmits TIMIT reference transcripts "
                 "(LDC93S1, licensed) to a third-party API; confirm your LDC terms permit it.")

submitted = json.load(open(BATCH_JSON)) if os.path.exists(BATCH_JSON) else {}

for judge in JUDGES:
    done = submitted.setdefault(judge, {})
    for start in range(0, len(full), CHUNK):
        key = str(start)
        if key in done:
            print(f"{judge} rows {start}-{start+CHUNK}: already submitted ({done[key]})")
            continue
        reqs = []
        for i in range(start, min(start + CHUNK, len(full))):
            r = full[i]
            reqs.append({"custom_id": f"r{i}",
                         "params": params_for(judge, REF[r["path"]], normalizer(r["text"]))})
        batch = client.messages.batches.create(requests=reqs)
        done[key] = batch.id
        with open(BATCH_JSON, "w") as f:          # persist before the next chunk
            json.dump(submitted, f, indent=1)
        print(f"{judge} rows {start}-{start+len(reqs)}: {batch.id}")

print(f"\n{sum(len(v) for v in submitted.values())} batches on record -> {BATCH_JSON}")

## 6. Poll and collect

Batches usually finish inside an hour; the ceiling is 24. This cell polls every batch to `ended`,
then streams the results.

Two details the Batch API makes easy to get wrong. **Results arrive in arbitrary order** — they are
keyed by `custom_id`, never by position. And **per-request failures are values, not exceptions**:
each result carries a `result.type` of `succeeded` / `errored` / `canceled` / `expired`, so a
handful of failures inside an otherwise fine batch will pass silently unless counted. They are
counted below and asserted to be zero before anything downstream runs.

Cache effectiveness is reported from the returned `usage` rather than assumed — if
`cache_read_input_tokens` is flat zero on an arm that section 4 said would cache, the prefix is
being invalidated somewhere and the run cost more than projected.

In [ ]:
submitted = json.load(open(BATCH_JSON))
VERDICT, usage_tot, fails = {}, collections.Counter(), collections.Counter()

for judge, chunks in submitted.items():
    for key, bid in sorted(chunks.items(), key=lambda kv: int(kv[0])):
        while True:
            b = client.messages.batches.retrieve(bid)
            if b.processing_status == "ended":
                break
            print(f"  {judge} {bid}: {b.processing_status} "
                  f"(processing {b.request_counts.processing})", flush=True)
            time.sleep(60)
        for res in client.messages.batches.results(bid):
            i = int(res.custom_id[1:])
            if res.result.type != "succeeded":
                fails[(judge, res.result.type)] += 1
                continue
            msg = res.result.message
            text = next(b.text for b in msg.content if b.type == "text")
            VERDICT[(judge, i)] = json.loads(text)["label"]
            u = msg.usage
            usage_tot[(judge, "in")]         += u.input_tokens
            usage_tot[(judge, "out")]        += u.output_tokens
            usage_tot[(judge, "cache_read")] += getattr(u, "cache_read_input_tokens", 0) or 0
        print(f"{judge} chunk {key}: collected ({bid})")

assert not fails, f"non-succeeded results: {dict(fails)}"
for judge in JUDGES:
    n = sum(1 for (j, _) in VERDICT if j == judge)
    assert n == len(full), f"{judge}: {n} verdicts for {len(full)} rows"
assert set(VERDICT.values()) <= set(LABELS), set(VERDICT.values()) - set(LABELS)

print(f"\n{len(VERDICT)} verdicts, no failures\n")
hdr = f"{'judge':>18} {'uncached in':>12} {'cache read':>11} {'out':>9} {'cache hit %':>12}"
print(hdr + "\n" + "-" * len(hdr))
for j in JUDGES:
    i_, c_, o_ = (usage_tot[(j, k)] for k in ("in", "cache_read", "out"))
    print(f"{j:>18} {i_:>12} {c_:>11} {o_:>9} {100*c_/max(1, i_+c_):>11.1f}%")

## 7. Hallucination Error Rate

HER is Atwany et al.'s metric: hallucination errors over total examples in the cell. It is directly
comparable to the taxonomy's hallucination rate — same denominator, same 1000 clips — which is what
makes section 8's agreement analysis possible at all.

Wilson intervals again, and the speaker-clustered bootstrap on the headline condition, for the same
reason as everywhere else in this project: 168 speakers contribute 5–6 clips each, so utterances are
not independent.

In [ ]:
def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def cluster_ci(flags, speakers, n_boot=10000, alpha=0.05, seed=0):
    f, g = np.asarray(flags, float), np.asarray(speakers)
    pool = [np.flatnonzero(g == u) for u in np.unique(g)]
    K, rng = len(pool), np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        boots[b] = f[np.concatenate([pool[j] for j in rng.integers(0, K, K)])].mean()
    return float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


rows = []
for i, r in enumerate(full):
    key = (r["model"], r["path"], int(r["offset_s"]), r["timestamps"])
    t = TAX[key]
    row = {"model": r["model"], "path": r["path"], "speaker": t["speaker"],
           "region": t["region"], "offset_s": int(r["offset_s"]),
           "timestamps": r["timestamps"], "taxonomy": t["category"],
           "taxonomy_halluc": int(t["halluc"])}
    for j in JUDGES:
        row[f"verdict_{j}"]  = VERDICT[(j, i)]
        row[f"halluc_{j}"]   = int(VERDICT[(j, i)] == "Hallucination Error")
    rows.append(row)

BY = collections.defaultdict(list)
for r in rows:
    BY[(r["model"], r["offset_s"], r["timestamps"])].append(r)

SUMMARY = {}
for j in JUDGES:
    print(f"\nHER (%) — judge {j}   [Wilson 95% CI]\n")
    hdr = f"{'model':>9} {'params':>8} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS)
    print(hdr + "\n" + "-" * len(hdr))
    for m in MODELS:
        out = []
        for o, t in CONDS:
            rs = BY[(m, o, t)]
            k = sum(r[f"halluc_{j}"] for r in rs)
            p, lo, hi = wilson(k, len(rs))
            SUMMARY[f"{j}|{m}|{o}|{t}"] = {"n": len(rs), "k": k, "her": p,
                                           "wilson_lo": lo, "wilson_hi": hi}
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} {PARAMS[m]:>8} " + " ".join(f"{c:>21}" for c in out))

print(f"\ntaxonomy hallucination rate (%) for comparison\n")
hdr = f"{'model':>9} " + " ".join(f"{f'{o} s / ts {t}':>14}" for o, t in CONDS)
print(hdr + "\n" + "-" * len(hdr))
for m in MODELS:
    out = []
    for o, t in CONDS:
        rs = BY[(m, o, t)]
        p, lo, hi = wilson(sum(r["taxonomy_halluc"] for r in rs), len(rs))
        SUMMARY[f"taxonomy|{m}|{o}|{t}"] = {"n": len(rs), "her": p,
                                            "wilson_lo": lo, "wilson_hi": hi}
        out.append(f"{100*p:5.1f}")
    print(f"{m:>9} " + " ".join(f"{c:>14}" for c in out))

print(f"\nspeaker-clustered cross-check at {HEAD[0]} s / ts {HEAD[1]}")
for j in JUDGES + ["taxonomy"]:
    col = "taxonomy_halluc" if j == "taxonomy" else f"halluc_{j}"
    for m in MODELS:
        rs = BY[(m, *HEAD)]
        lo, hi = cluster_ci([r[col] for r in rs], [r["speaker"] for r in rs])
        SUMMARY[f"{j}|{m}|{HEAD[0]}|{HEAD[1]}"]["cluster_lo"] = lo
        SUMMARY[f"{j}|{m}|{HEAD[0]}|{HEAD[1]}"]["cluster_hi"] = hi
print("  stored in SUMMARY; see the provenance file")

## 8. Agreement — the number this notebook exists to produce

Atwany et al. report raw agreement between evaluation methods: human–human 0.71, human–GPT 0.60,
human–Gemini 0.59, GPT–Gemini 0.78, and human–heuristic **0.00** against a threshold heuristic
(cosine similarity, WER, and perplexity cuts) not unlike ours.

That 0.00 is the reason to run this. If our taxonomy agrees with the LLM judges at anything
approaching their cross-model figure, the "heuristics don't work" result does not transfer to this
detector, and we can say so with a number instead of an argument. If it agrees at near zero, that is
a finding about our detector that is far better to discover here than in review.

Two quantities, because agreement alone is ambiguous at low base rates: **raw agreement** over all
20 000 rows (dominated by the ~98% both methods call clean) and **Cohen's kappa**, which corrects
for chance and is the honest figure when one class is rare. Positive-class agreement — the share of
rows either method flags that both flag — is printed alongside, since that is where the two methods
actually have to decide something.

In [ ]:
def agree(a, b):
    """Raw agreement, Cohen's kappa, and Jaccard over the positive class."""
    a, b = np.asarray(a, int), np.asarray(b, int)
    n = len(a)
    po = float((a == b).mean())
    pe = ((a.mean() * b.mean()) + ((1 - a.mean()) * (1 - b.mean())))
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    both, either = int((a & b).sum()), int((a | b).sum())
    return po, kappa, (both / either if either else float("nan")), both, either


import itertools

COL = {j: f"halluc_{j}" for j in JUDGES}
COL["taxonomy"] = "taxonomy_halluc"
PAIRS = list(itertools.combinations(["taxonomy"] + JUDGES, 2))   # every pair, any judge count

print(f"agreement over all {len(rows)} rows\n")
hdr = f"{'pair':>44} {'raw':>7} {'kappa':>7} {'jaccard':>8} {'both':>6} {'either':>7}"
print(hdr + "\n" + "-" * len(hdr))
AGREE = {}
for x, y in PAIRS:
    po, k, jac, both, either = agree([r[COL[x]] for r in rows], [r[COL[y]] for r in rows])
    AGREE[f"{x} vs {y}"] = {"raw": po, "kappa": k, "jaccard": jac,
                            "both": both, "either": either}
    print(f"{f'{x} vs {y}':>44} {po:>7.3f} {k:>7.3f} {jac:>8.3f} {both:>6} {either:>7}")

print("\nAtwany et al. Table 4 for reference:")
print("  human-human 0.71 | human-GPT 0.60 | human-Gemini 0.59 | GPT-Gemini 0.78")
print("  human-heuristic 0.00 | GPT-heuristic 0.10 | heuristic-Gemini 0.14")
print("  (their heuristic: cosine 0.2 + WER 30 + Flan-T5 perplexity 200 -- a different"
      "\n   feature set from ours, but the same kind of instrument)")

print(f"\nwhere the taxonomy and {JUDGES[0]} disagree, by taxonomy category "
      f"({HEAD[0]} s / ts {HEAD[1]})\n")
hdr = f"{'taxonomy category':>14} {'n':>6} {'llm halluc':>11} {'tax halluc':>11} {'disagree':>9}"
print(hdr + "\n" + "-" * len(hdr))
for cat in ["faithful", "errorful", "truncated", "empty", "degenerate", "runaway", "untethered"]:
    rs = [r for m in MODELS for r in BY[(m, *HEAD)] if r["taxonomy"] == cat]
    if not rs:
        continue
    dis = sum(r[COL["taxonomy"]] != r[COL[JUDGES[0]]] for r in rs)
    print(f"{cat:>14} {len(rs):>6} {sum(r[COL[JUDGES[0]]] for r in rs):>11}"
          f" {sum(r[COL['taxonomy']] for r in rs):>11} {dis:>9}")

print("\n`degenerate` is the row to read first: Atwany et al. classify repetition as"
      "\nOscillation Error -- a NON-hallucination -- so the judges should disagree with our"
      "\ntaxonomy there by construction. That is a definitional split, not a detector failure.")

## 9. Write the results

Git-safe by construction: every column is an identifier or a category name drawn from a fixed
label set. There is no free text to leak, so this file needs no Drive-only companion — the same
property `halluc_taxonomy.csv` has.

The provenance records the licensing decision explicitly. A later reader should be able to see that
transmitting reference text was chosen deliberately, with the grid size it applied to, rather than
having to infer it from the fact that the notebook exists.

In [ ]:
SAFE_FIELDS = (["model", "path", "speaker", "region", "offset_s", "timestamps",
                "taxonomy", "taxonomy_halluc"]
               + [f"verdict_{j}" for j in JUDGES] + [f"halluc_{j}" for j in JUDGES])
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript"}
IDENT = ({"model", "path", "speaker", "region", "timestamps", "taxonomy"}
         | {f"verdict_{j}" for j in JUDGES})

assert not (set(SAFE_FIELDS) & FORBIDDEN), "a text-bearing column leaked into SAFE_FIELDS"
assert set(rows[0]) == set(SAFE_FIELDS), set(rows[0]) ^ set(SAFE_FIELDS)
for row in rows:
    for k, v in row.items():
        assert k in IDENT or " " not in str(v), (k, v)
    for j in JUDGES:
        assert row[f"verdict_{j}"] in LABELS, row[f"verdict_{j}"]

with open(OUT_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=SAFE_FIELDS)
    wr.writeheader()
    wr.writerows(rows)

PROVENANCE = {
    "experiment": "LLM-based hallucination classification (Atwany et al. 2025, ACL Findings) "
                  "over experiment A's hypotheses, as an independent check on the text taxonomy",
    "method": {
        "prompt": "Atwany et al. Figure 5 (coarse-grained), transcribed verbatim",
        "grain": GRAIN, "labels": LABELS,
        "deviations": [
            "instructions in `system`, reference/hypothesis pair in the user turn, so the "
            "instruction block is a cacheable stable prefix; rendered text and order unchanged",
            "output_config.format with an enum schema replaces the prompt's 'produce only the "
            "classification' instruction, removing the parse step and preamble/XML failure modes",
            "thinking disabled on the opus-5 arm (matches their non-thinking GPT-4o-mini "
            "baseline); temperature=0 greedy decoding on the haiku-4-5 arm only, because "
            "opus-5 rejects the parameter",
        ],
        "judges": JUDGES, "batch_api": True, "max_tokens": MAX_TOKENS,
    },
    "licensing": {
        "decision": "full-grid submission chosen deliberately by the author",
        "consequence": "all 1000 TIMIT (LDC93S1) reference transcripts and 20000 hypotheses "
                       "were transmitted to the Anthropic API",
        "note": "every other notebook in this project keeps reference text off the wire; this "
                "one does not, and the trade was made knowingly rather than overlooked",
    },
    "derived_from": {"inputs": INPUT_DIGESTS, "reference_digest": rebuilt,
                     "batches": submitted},
    "packages": {"python": platform.python_version(), "anthropic": anthropic.__version__,
                 "numpy": np.__version__},
    "grid": {"models": MODELS, "conditions": [f"{o}s/ts-{t}" for o, t in CONDS],
             "clips_per_cell": 1000, "classifications": len(rows) * len(JUDGES)},
    "usage": {f"{j}|{k}": v for (j, k), v in usage_tot.items()},
    "agreement": AGREE,
    "summary": SUMMARY,
    "outputs": {"llm_verdict_per_utterance.csv": {"rows": len(rows), "fields": SAFE_FIELDS,
                                                  "git_safe": True}},
}
with open(PROV_JSON, "w") as f:
    json.dump(PROVENANCE, f, indent=1)

_back = list(csv.DictReader(open(OUT_CSV, newline="")))
assert len(_back) == len(rows) and set(_back[0]) == set(SAFE_FIELDS)

print(f"per-utterance -> {OUT_CSV}   ({len(rows)} rows, labels only; safe to commit)")
print(f"provenance    -> {PROV_JSON}   (records the licensing decision)")
print("\nchecked: no text-bearing column, every verdict drawn from the fixed label set")

## 10. Figure

The question the figure has to answer is whether two independent methods see the same shape across
scale — so both go on one axis, per arm, and the reader compares slopes rather than levels. Levels
will differ (the methods disagree about repetition by construction); a shared slope is the claim.

Two panels for the two timestamp arms, because the flat off-arm is what makes the on-arm meaningful.
Vector PDF at `pdf.fonttype = 42`, downloaded to local disk alongside the PNG.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e5e2"
CCAT   = ["#2a78d6", "#7a9a3b"]                       # fixed order, validated against SURFACE
SERIES = {"taxonomy": ("#eb6834", "text taxonomy")}
for n, j in enumerate(JUDGES):
    SERIES[j] = (CCAT[n % len(CCAT)], f"LLM HER ({j})")

x = np.arange(len(MODELS))
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), dpi=160, sharey=True)
fig.patch.set_facecolor(SURFACE)

for ax, arm in zip(axes, ("on", "off")):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for src, (colour, label) in SERIES.items():
        s   = [SUMMARY[f"{src}|{m}|{HEAD[0]}|{arm}"] for m in MODELS]
        mid = [100 * v["her"] for v in s]
        err = [[100 * (v["her"] - v["wilson_lo"]) for v in s],
               [100 * (v["wilson_hi"] - v["her"]) for v in s]]
        ax.errorbar(x, mid, yerr=err, fmt="o", color=colour, markersize=7, linewidth=2,
                    markeredgecolor=SURFACE, markeredgewidth=2, capsize=4, zorder=3,
                    label=label if arm == "off" else None)
        ax.plot(x, mid, color=colour, linewidth=2, zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{m}\n{PARAMS[m]}" for m in MODELS])
    ax.set_xlim(-0.5, len(MODELS) - 0.5)
    ax.set_title(f"timestamps {arm}", fontsize=12, color=INK, pad=10, loc="left")
    ax.tick_params(colors=INK2, labelsize=10, length=0)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRID); ax.spines[sp].set_linewidth(1)

axes[0].set_ylabel(f"flagged at {HEAD[0]} s (%)", fontsize=11, color=INK2)
leg = axes[1].legend(frameon=False, fontsize=9.5, loc="upper right", title="detector")
leg.get_title().set_color(INK2); leg.get_title().set_fontsize(9)
for t in leg.get_texts():
    t.set_color(INK2)
fig.suptitle("Two independent detectors, one scale trend "
             f"(utterances at {HEAD[0]} s, n = 1000 per point)",
             fontsize=12.5, color=INK, x=0.007, ha="left", y=1.02)
fig.tight_layout()

for ext in ("pdf", "png"):
    fig.savefig(f"llm_vs_taxonomy.{ext}", bbox_inches="tight", facecolor=SURFACE)
    fig.savefig(os.path.join(DRIVE_ROOT, f"llm_vs_taxonomy.{ext}"),
                bbox_inches="tight", facecolor=SURFACE)
print("wrote llm_vs_taxonomy.pdf and .png (+ copies on Drive)")
plt.show()

from google.colab import files
files.download("llm_vs_taxonomy.pdf")
files.download("llm_vs_taxonomy.png")

## 11. Standalone reload

No API key, no TIMIT, no Drive-only file, no earlier cell. Reads the committed
`llm_verdict_per_utterance.csv` and rebuilds both the HER table and the agreement figures — which
proves the git-safe file alone carries the whole result, and that nothing here depends on re-running
a paid API call.

In [ ]:
# --- standalone: run this alone in a fresh CPU runtime -------------------------------
import csv, collections, math, os
import numpy as np

PATH = "/content/drive/MyDrive/NAACL/llm_verdict_per_utterance.csv"   # or a local download
rows = list(csv.DictReader(open(PATH, newline="")))

MODELS_ = ["tiny", "base", "small", "medium", "large-v3"]
CONDS_  = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]
JUDGES_ = [c[len("halluc_"):] for c in rows[0] if c.startswith("halluc_")
           and c != "halluc_taxonomy"]

by = collections.defaultdict(list)
for r in rows:
    by[(r["model"], int(r["offset_s"]), r["timestamps"])].append(r)


def wilson_(k, n, z=1.959963985):
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


print(f"reloaded {len(rows)} rows | judges: {', '.join(JUDGES_)}\n")
for src in ["taxonomy"] + JUDGES_:
    col = "taxonomy_halluc" if src == "taxonomy" else f"halluc_{src}"
    print(f"{src}")
    print(f"{'model':>9} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS_))
    for m in MODELS_:
        out = []
        for o, t in CONDS_:
            rs = by[(m, o, t)]
            p, lo, hi = wilson_(sum(int(r[col]) for r in rs), len(rs))
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} " + " ".join(f"{c:>21}" for c in out))
    print()

print("agreement (raw / kappa / jaccard)")
cols = ["taxonomy_halluc"] + [f"halluc_{j}" for j in JUDGES_]
for i in range(len(cols)):
    for k in range(i + 1, len(cols)):
        a = np.array([int(r[cols[i]]) for r in rows])
        b = np.array([int(r[cols[k]]) for r in rows])
        po = float((a == b).mean())
        pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
        kap = (po - pe) / (1 - pe) if pe < 1 else float("nan")
        jac = (a & b).sum() / max(1, (a | b).sum())
        print(f"  {cols[i]:>22} vs {cols[k]:<22} {po:.3f} / {kap:.3f} / {jac:.3f}")